# NeuralGeoQA — Training Notebook

Two-phase training on a **T4 GPU** Colab instance.

| Phase | Data | Heads trained | Output |
|-------|------|---------------|--------|
| 1 | 19,481 Wikidata SimpleQuestions | Span + Relation | `phase1_output/best_model.pt` |
| 2 | 1,087 GeoQuestions1089 | QType only (Span+Relation frozen) | `qtype_model_v2/best_model.pt` |

Run cells top to bottom. Phase 1 must complete before Phase 2.

In [ ]:
# Cell 1 — Install
!pip install -q transformers==4.40.0 torch scikit-learn pandas tqdm flair scipy requests fuzzywuzzy python-Levenshtein accelerate

In [ ]:
# Cell 2 — Mount Drive + paths
from google.colab import drive
drive.mount('/content/drive')

import os

ANS_DIR  = '/content/drive/MyDrive/ANS/triple_head/'
P1_TRAIN = os.path.join(ANS_DIR, 'train/flair_train_model_4.txt')
P1_VALID = os.path.join(ANS_DIR, 'valid/flair_valid_model_4.txt')
P1_TEST  = os.path.join(ANS_DIR, 'test/flair_test_model_4.txt')
P1_OUT   = os.path.join(ANS_DIR, 'phase1_output')

GEO_DIR  = '/content/drive/MyDrive/triple_head/GEO/'
P2_TRAIN = os.path.join(GEO_DIR, 'geo_train.tsv')
P2_OUT   = os.path.join(GEO_DIR, 'qtype_model_v2')

for d in [P1_OUT, os.path.join(P1_OUT,'final_model'), P2_OUT]:
    os.makedirs(d, exist_ok=True)

assert os.path.exists(P1_TRAIN), f'Missing: {P1_TRAIN} — run preprocess.py first'
assert os.path.exists(P2_TRAIN), f'Missing: {P2_TRAIN}'
print('Paths OK.')

In [ ]:
# Cell 3 — Shared: imports, GeoQABERT, class merge map

import os, re, gc, json, time
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import torch
import torch.nn as nn
import pandas as pd
from collections import Counter
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP    = DEVICE.type == 'cuda'
MODEL_NAME = 'bert-base-uncased'
MAX_LEN    = 96
print(f'Device: {DEVICE}  AMP: {USE_AMP}')

# ── Class merge map (12 fine-grained → 8 merged) ─────────────────────────────
CLASS_MERGE_MAP = {
    'C_class_spatial_rel': 'C_class',
    'C_class_in':          'C_class',
    'B_directional':       'B_spatial',
    'E_class_distance':    'E_class_near',
    'G_comparative':       'G_superlative',
    'A_attribute':         'A_attribute',
    'B_boolean':           'B_boolean',
    'B_spatial':           'B_spatial',
    'E_class_near':        'E_class_near',
    'F_thematic_spatial':  'F_thematic_spatial',
    'G_count':             'G_count',
    'G_superlative':       'G_superlative',
}
def merge_qtype(q): return CLASS_MERGE_MAP.get(q, q)

# ── GeoQABERT: single encoder, three heads ───────────────────────────────────
class GeoQABERT(nn.Module):
    """
    bert-base-uncased with span, relation, and qtype heads.
    Phase 1: trains span + relation. Phase 2: freezes those, trains qtype.
    """
    def __init__(self, model_name, num_relations, num_qtypes,
                 dropout=0.1, class_weights=None):
        super().__init__()
        self.config           = AutoConfig.from_pretrained(model_name)
        self.encoder          = AutoModel.from_pretrained(model_name)
        h                     = self.config.hidden_size
        self.span_start       = nn.Linear(h, 1)
        self.span_end         = nn.Linear(h, 1)
        self.rel_dropout      = nn.Dropout(dropout)
        self.rel_classifier   = nn.Linear(h, num_relations)
        self.qtype_dropout    = nn.Dropout(dropout)
        self.qtype_classifier = nn.Linear(h, num_qtypes)
        self.qtype_loss_fn    = nn.CrossEntropyLoss(weight=class_weights) if class_weights is not None else nn.CrossEntropyLoss()
        self.span_loss_fn     = nn.CrossEntropyLoss()
        self.rel_loss_fn      = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask,
                start_positions=None, end_positions=None,
                relation_labels=None, qtype_labels=None):
        out      = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        seq_out  = out.last_hidden_state
        cls_out  = seq_out[:, 0, :]
        sl       = self.span_start(seq_out).squeeze(-1)
        el       = self.span_end(seq_out).squeeze(-1)
        rl       = self.rel_classifier(self.rel_dropout(cls_out))
        ql       = self.qtype_classifier(self.qtype_dropout(cls_out))
        loss = None
        if start_positions is not None:
            loss = (self.span_loss_fn(sl, start_positions)
                  + self.span_loss_fn(el, end_positions)
                  + self.rel_loss_fn(rl, relation_labels))
        elif qtype_labels is not None:
            loss = self.qtype_loss_fn(ql, qtype_labels)
        return {'loss':loss,'start_logits':sl,'end_logits':el,'rel_logits':rl,'qtype_logits':ql}

print('GeoQABERT defined.')

---
## Phase 1 — Pretrain on Wikidata SimpleQuestions
Trains span (subject entity start/end tokens) and relation (125 Wikidata properties) heads.
The encoder learns general question understanding over Wikidata.

In [ ]:
# Cell 4 — Phase 1: span helpers + dataset

def whitespace_token_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r'\S+', text)]

def word_span_to_char_span(question, sw, ew, subject=None):
    if question is None: return None, None
    q = str(question); spans = whitespace_token_spans(q)
    if isinstance(sw,int) and isinstance(ew,int) and 0<=sw<ew<=len(spans):
        return spans[sw][0], spans[ew-1][1]
    if subject:
        s=str(subject).strip(); pos=q.lower().find(s.lower())
        if pos!=-1: return pos, pos+len(s)
    return None, None

def char_span_to_token_span(om, cs, ce):
    if cs is None or ce is None: return 0, 0
    st=et=None
    for i,(s,e) in enumerate(om):
        if s==0 and e==0: continue
        if st is None and s<=cs<e: st=i
        if s<ce<=e: et=i
    if st is None:
        for i,(s,e) in enumerate(om):
            if s==0 and e==0: continue
            if s>=cs: st=i; break
    if et is None:
        for i in range(len(om)-1,-1,-1):
            s,e=om[i]
            if s==0 and e==0: continue
            if e<=ce: et=i; break
    if st is None or et is None or st>et: return 0,0
    return int(st), int(et)

class SimpleQuestionsDataset(Dataset):
    def __init__(self, data, tokenizer, relation_to_id, max_length=96):
        self.data=data; self.tok=tokenizer
        self.rel_map=relation_to_id; self.max_len=max_length
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        r=self.data.iloc[i]
        inp=self.tok(str(r[6]),add_special_tokens=True,max_length=self.max_len,
                     padding='max_length',truncation=True,return_tensors='pt',
                     return_offsets_mapping=True)
        ids=inp['input_ids'].squeeze(0); mask=inp['attention_mask'].squeeze(0)
        om=inp['offset_mapping'].squeeze(0).tolist()
        cs,ce=word_span_to_char_span(str(r[6]),int(r[7]),int(r[8]),str(r[0]))
        ts,te=char_span_to_token_span(om,cs,ce)
        seq=int(ids.shape[0])
        ts=max(0,min(ts,seq-1)); te=max(0,min(te,seq-1))
        if ts>te: ts=te=0
        return {'input_ids':ids,'attention_mask':mask,
                'start_positions':torch.tensor(ts,dtype=torch.long),
                'end_positions':  torch.tensor(te,dtype=torch.long),
                'relation_label': torch.tensor(self.rel_map[r[3]],dtype=torch.long)}

print('Phase 1 dataset ready.')

In [ ]:
# Cell 5 — Phase 1: build relation vocab + loaders

P1_EPOCHS = 3
P1_BATCH  = 16
P1_LR     = 2e-5

tok_p1 = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

all_data = pd.concat([pd.read_csv(f,sep='\t',header=None) for f in [P1_TRAIN,P1_VALID,P1_TEST]])
unique_rels    = all_data[3].unique()
relation_to_id = {r:i for i,r in enumerate(unique_rels)}
id_to_relation = {i:r for r,i in relation_to_id.items()}
NUM_RELATIONS  = len(relation_to_id)
print(f'Relations: {NUM_RELATIONS}')

def make_p1_loader(path, shuffle):
    df=pd.read_csv(path,sep='\t',header=None)
    ds=SimpleQuestionsDataset(df,tok_p1,relation_to_id,MAX_LEN)
    return DataLoader(ds,batch_size=P1_BATCH,shuffle=shuffle,pin_memory=True,num_workers=0)

p1_tr = make_p1_loader(P1_TRAIN, True)
p1_vl = make_p1_loader(P1_VALID, False)
p1_ts = make_p1_loader(P1_TEST,  False)
print(f'Train: {len(p1_tr)} batches  Valid: {len(p1_vl)}')

In [ ]:
# Cell 6 — Phase 1: train + eval loop

gc.collect(); torch.cuda.empty_cache() if USE_AMP else None

model_p1    = GeoQABERT(MODEL_NAME, NUM_RELATIONS, num_qtypes=1).to(DEVICE)
opt_p1      = AdamW(model_p1.parameters(), lr=P1_LR)
total_steps = len(p1_tr) * P1_EPOCHS
sched_p1    = get_linear_schedule_with_warmup(opt_p1,
    num_warmup_steps=int(0.06*total_steps), num_training_steps=total_steps)
scaler_p1   = torch.amp.GradScaler('cuda', enabled=USE_AMP)

@torch.no_grad()
def p1_eval(loader):
    model_p1.eval(); preds,trues,sc,ec,n=[],[],0,0,0
    for b in tqdm(loader, desc='Eval', leave=False):
        ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE)
        st=b['start_positions'].to(DEVICE); en=b['end_positions'].to(DEVICE)
        rl=b['relation_label'].to(DEVICE)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            out=model_p1(input_ids=ids,attention_mask=mask,
                         start_positions=st,end_positions=en,relation_labels=rl)
        sp=torch.argmax(out['start_logits'],1); ep=torch.argmax(out['end_logits'],1)
        rp=torch.argmax(out['rel_logits'],1)
        bs=sp.size(0); n+=bs
        sc+=int((sp==st).sum()); ec+=int((ep==en).sum())
        preds.extend(rp.cpu().tolist()); trues.extend(rl.cpu().tolist())
    return {'start_acc':sc/n,'end_acc':ec/n,
            'rel_acc':sum(p==t for p,t in zip(preds,trues))/len(preds),
            'rel_f1_w':f1_score(trues,preds,average='weighted',zero_division=0)}

best_rel_acc = 0.0
t0 = time.time()

for epoch in range(1, P1_EPOCHS+1):
    model_p1.train(); total_loss=0.0; opt_p1.zero_grad(set_to_none=True)
    for step,b in enumerate(tqdm(p1_tr, desc=f'P1 Epoch {epoch}')):
        ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE)
        st=b['start_positions'].to(DEVICE); en=b['end_positions'].to(DEVICE)
        rl=b['relation_label'].to(DEVICE)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            out=model_p1(input_ids=ids,attention_mask=mask,
                         start_positions=st,end_positions=en,relation_labels=rl)
        scaler_p1.scale(out['loss']).backward()
        scaler_p1.step(opt_p1); scaler_p1.update()
        sched_p1.step(); opt_p1.zero_grad(set_to_none=True)
        total_loss += out['loss'].item()
    vm=p1_eval(p1_vl)
    print(f'Epoch {epoch}  loss={total_loss/len(p1_tr):.4f}  '
          f'StartAcc={vm["start_acc"]:.4f}  EndAcc={vm["end_acc"]:.4f}  '
          f'RelAcc={vm["rel_acc"]:.4f}  RelF1={vm["rel_f1_w"]:.4f}')
    if vm['rel_acc'] > best_rel_acc:
        best_rel_acc = vm['rel_acc']
        torch.save(model_p1.state_dict(), os.path.join(P1_OUT,'best_model.pt'))
        print(f'  New best RelAcc={best_rel_acc:.4f}')
    if USE_AMP: torch.cuda.empty_cache()

print(f'Phase 1 done in {time.time()-t0:.1f}s')

In [ ]:
# Cell 7 — Phase 1: test evaluation + save

import shutil
model_p1.load_state_dict(torch.load(
    os.path.join(P1_OUT,'best_model.pt'), map_location=DEVICE, weights_only=True))
tm=p1_eval(p1_ts)
print(f'Test  StartAcc={tm["start_acc"]:.4f}  EndAcc={tm["end_acc"]:.4f}  '
      f'RelAcc={tm["rel_acc"]:.4f}  RelF1={tm["rel_f1_w"]:.4f}')

final_dir = os.path.join(P1_OUT,'final_model')
os.makedirs(final_dir, exist_ok=True)
shutil.copy(os.path.join(P1_OUT,'best_model.pt'), os.path.join(final_dir,'best_model.pt'))
tok_p1.save_pretrained(os.path.join(P1_OUT,'tokenizer'))

p1_cfg = {'model_name':MODEL_NAME,'num_relations':NUM_RELATIONS,'num_qtypes':1,
           'max_length':MAX_LEN,
           'relation_to_id':relation_to_id,
           'id_to_relation':{str(k):v for k,v in id_to_relation.items()},
           'best_rel_acc':best_rel_acc,'test_rel_acc':tm['rel_acc']}
with open(os.path.join(final_dir,'config.json'),'w') as f: json.dump(p1_cfg,f,indent=2)
print(f'Phase 1 saved to {P1_OUT}')

---
## Phase 2 — Fine-tune QType head on GeoQuestions1089

Loads the Phase 1 encoder. **Span + relation heads are frozen.**
Only the QType head (+ encoder at a lower LR) is trained.

Class imbalance handled by:
1. Merging 12 types → 8 classes (types with <10 examples absorbed into parent)
2. Weighted cross-entropy (rare classes weighted more heavily)

Primary metric: **macro F1** — equal weight per class, better than accuracy on imbalanced data.

At inference, merged predictions are split back to 12 types via keyword rules in `evaluate.py`.

In [ ]:
# Cell 8 — Phase 2: load GeoQuestions, merge classes, compute weights

df_geo = pd.read_csv(P2_TRAIN, sep='\t')
print(f'Loaded {len(df_geo)} GeoQuestions examples')

df_geo['QType_Merged'] = df_geo['QType'].map(merge_qtype)

print('\n=== Before merging (12 classes) ===')
for k,v in df_geo['QType'].value_counts().items(): print(f'  {k:<25} {v:>4}')
print('\n=== After merging (8 classes) ===')
for k,v in df_geo['QType_Merged'].value_counts().items(): print(f'  {k:<25} {v:>4}')

qtypes      = sorted(df_geo['QType_Merged'].unique())
qtype_to_id = {q:i for i,q in enumerate(qtypes)}
id_to_qtype = {i:q for q,i in qtype_to_id.items()}
NUM_QTYPES  = len(qtype_to_id)
print(f'\nMerged classes: {NUM_QTYPES}')

counts  = Counter(df_geo['QType_Merged'])
raw_w   = [1.0/counts[id_to_qtype[i]] for i in range(NUM_QTYPES)]
total_w = sum(raw_w)
weights = [w*NUM_QTYPES/total_w for w in raw_w]
class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

print('\n=== Class weights ===')
for i in range(NUM_QTYPES):
    print(f'  {id_to_qtype[i]:<25} count={counts[id_to_qtype[i]]:>4}  weight={weights[i]:.3f}')

In [ ]:
# Cell 9 — Phase 2: train/val split + dataloaders

P2_EPOCHS   = 15
P2_BATCH    = 16
P2_LR_ENC  = 2e-5   # encoder: low LR to preserve Phase 1 representations
P2_LR_HEAD = 3e-4   # qtype head: higher LR, learning from scratch
P2_WARMUP  = 0.1
P2_VALSPLIT = 0.15

train_df, val_df = train_test_split(
    df_geo, test_size=P2_VALSPLIT, stratify=df_geo['QType_Merged'], random_state=42)
print(f'Train: {len(train_df)}  Val: {len(val_df)}')

tok_p2 = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

class QTypeDataset(Dataset):
    def __init__(self, df, tokenizer, qtype_to_id, max_length=96):
        self.df=df.reset_index(drop=True); self.tok=tokenizer
        self.q2i=qtype_to_id; self.ml=max_length
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r=self.df.iloc[i]
        enc=self.tok(str(r['Question']),add_special_tokens=True,max_length=self.ml,
                     padding='max_length',truncation=True,return_tensors='pt')
        return {'input_ids':enc['input_ids'].squeeze(0),
                'attention_mask':enc['attention_mask'].squeeze(0),
                'qtype_label':torch.tensor(self.q2i[str(r['QType_Merged'])],dtype=torch.long)}

p2_tr = DataLoader(QTypeDataset(train_df,tok_p2,qtype_to_id,MAX_LEN),
    batch_size=P2_BATCH, shuffle=True,  pin_memory=True, num_workers=0)
p2_vl = DataLoader(QTypeDataset(val_df,  tok_p2,qtype_to_id,MAX_LEN),
    batch_size=P2_BATCH, shuffle=False, pin_memory=True, num_workers=0)
print(f'Train: {len(p2_tr)} batches  Val: {len(p2_vl)}')

In [ ]:
# Cell 10 — Phase 2: build model, load Phase 1 weights, freeze heads

gc.collect(); torch.cuda.empty_cache() if USE_AMP else None

with open(os.path.join(P1_OUT,'final_model','config.json')) as f:
    p1_cfg = json.load(f)

model_p2 = GeoQABERT(MODEL_NAME, p1_cfg['num_relations'],
                     NUM_QTYPES, class_weights=class_weights).to(DEVICE)

ckpt    = os.path.join(P1_OUT,'final_model','best_model.pt')
state   = torch.load(ckpt, map_location=DEVICE, weights_only=True)
missing, unexpected = model_p2.load_state_dict(state, strict=False)
print(f'Loaded Phase 1 weights  missing={len(missing)} (new qtype head — expected)  unexpected={len(unexpected)}')

# Freeze span + relation heads
for head in [model_p2.span_start, model_p2.span_end,
             model_p2.rel_dropout, model_p2.rel_classifier]:
    for p in head.parameters(): p.requires_grad = False

trainable = sum(p.numel() for p in model_p2.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')

opt_p2 = AdamW([
    {'params': model_p2.encoder.parameters(),          'lr': P2_LR_ENC},
    {'params': model_p2.qtype_classifier.parameters(), 'lr': P2_LR_HEAD},
    {'params': model_p2.qtype_dropout.parameters(),    'lr': P2_LR_HEAD},
])
total_steps = len(p2_tr) * P2_EPOCHS
sched_p2    = get_linear_schedule_with_warmup(opt_p2,
    num_warmup_steps=int(P2_WARMUP*total_steps), num_training_steps=total_steps)
scaler_p2   = torch.amp.GradScaler('cuda', enabled=USE_AMP)
print(f'Encoder LR={P2_LR_ENC}  Head LR={P2_LR_HEAD}')

In [ ]:
# Cell 11 — Phase 2: training loop

@torch.no_grad()
def p2_eval(loader):
    model_p2.eval(); preds,trues,total=[],[],0.0
    for b in tqdm(loader, desc='Eval', leave=False):
        ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE)
        lbl=b['qtype_label'].to(DEVICE)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            out=model_p2(input_ids=ids,attention_mask=mask,qtype_labels=lbl)
        total+=out['loss'].item()
        preds.extend(torch.argmax(out['qtype_logits'],1).cpu().tolist())
        trues.extend(lbl.cpu().tolist())
    return {'loss':total/len(loader),
            'acc':sum(p==t for p,t in zip(preds,trues))/len(preds),
            'f1_w':f1_score(trues,preds,average='weighted',zero_division=0),
            'f1_m':f1_score(trues,preds,average='macro',zero_division=0),
            'preds':preds,'trues':trues}

best_f1m  = 0.0
best_ckpt = os.path.join(P2_OUT,'best_model.pt')
t0 = time.time()

for epoch in range(1, P2_EPOCHS+1):
    model_p2.train(); total_loss=0.0; opt_p2.zero_grad(set_to_none=True)
    for b in tqdm(p2_tr, desc=f'P2 Epoch {epoch}'):
        ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE)
        lbl=b['qtype_label'].to(DEVICE)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            out=model_p2(input_ids=ids,attention_mask=mask,qtype_labels=lbl)
        scaler_p2.scale(out['loss']).backward()
        scaler_p2.step(opt_p2); scaler_p2.update()
        sched_p2.step(); opt_p2.zero_grad(set_to_none=True)
        total_loss += out['loss'].item()
    vm=p2_eval(p2_vl)
    print(f'Epoch {epoch:>2}  loss={total_loss/len(p2_tr):.4f}  '
          f'acc={vm["acc"]:.4f}  F1-w={vm["f1_w"]:.4f}  F1-m={vm["f1_m"]:.4f} ←')
    if vm['f1_m'] > best_f1m:
        best_f1m = vm['f1_m']
        torch.save(model_p2.state_dict(), best_ckpt)
        print(f'  New best F1-macro={best_f1m:.4f}')
    if USE_AMP: torch.cuda.empty_cache()

print(f'Phase 2 done in {time.time()-t0:.1f}s  Best F1-macro={best_f1m:.4f}')

In [ ]:
# Cell 12 — Phase 2: final classification report + save

model_p2.load_state_dict(torch.load(best_ckpt, map_location=DEVICE, weights_only=True))
vm = p2_eval(p2_vl)

target_names = [id_to_qtype[i] for i in sorted(id_to_qtype)]
print('\n=== Best model — Validation report ===')
print(classification_report(vm['trues'], vm['preds'],
                             target_names=target_names, zero_division=0))

tok_p2.save_pretrained(os.path.join(P2_OUT,'tokenizer'))

config = {
    'model_name':      MODEL_NAME,
    'num_relations':   p1_cfg['num_relations'],
    'num_qtypes':      NUM_QTYPES,
    'max_length':      MAX_LEN,
    'qtype_to_id':     qtype_to_id,
    'id_to_qtype':     {str(k):v for k,v in id_to_qtype.items()},
    'relation_to_id':  p1_cfg['relation_to_id'],
    'id_to_relation':  p1_cfg['id_to_relation'],
    'class_merge_map': CLASS_MERGE_MAP,
    'class_weights':   weights,
    'best_f1_macro':   best_f1m,
}
with open(os.path.join(P2_OUT,'config.json'),'w') as f: json.dump(config,f,indent=2)

print(f'\nSaved to {P2_OUT}')
print(f'  best_model.pt  ← load this in NeuralGeoQA_Eval.ipynb')
print(f'  config.json    ← id_to_qtype, id_to_relation')
print(f'  tokenizer/')
print(f'\nBest val F1-macro: {best_f1m:.4f}')